In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import yaml
import sys
import dask
import socket
import calendar
import pandas as pd
import os

# import utilities
from rossby_utils import cds_sort_utils as cds_sort

import warnings
warnings.filterwarnings('ignore')
import gc

### Set up dask cluster.  Note, you will need to change
- local_directory
- project

In [2]:
from dask_jobqueue import PBSCluster
from dask.distributed import Client
dask.config.set({"distributed.scheduler.worker_saturation":1.0})
dask.config.set({"optimization.fuse.active": False})
dask.config.set({
    "distributed.worker.memory.target": 0.6,
    "distributed.worker.memory.spill": 0.7,
    "distributed.worker.memory.pause": 0.8,
    "distributed.worker.memory.terminate": 0.95,
})

cluster = PBSCluster(
    cores = 1,
    memory = '30GB',
    processes = 1,
    queue = 'casper',
    local_directory = '/glade/derecho/scratch/islas/dask_tmp/',
    resource_spec = 'select=1:ncpus=1:mem=30GB',
    project='P04010022',
    walltime='01:00:00',
    interface='mgt')

# scale up
cluster.scale(12)
#cluster.adapt(minimum=1, maximum=12)

# change your urls to the dask dashboard so that you can see it
hostname = socket.getfqdn()
dask.config.set({"distributed.dashboard.link":
        f"https://ondemand.hpc.ucar.edu/rnode/{hostname}/{{port}}/status"
})

client = Client(cluster)

#### Wait after the next cell until you get the workers

In [3]:
cluster

PBSCluster(9001a337, 'tcp://10.18.206.66:43661', workers=12, threads=12, memory=335.28 GiB)

#### Read in system information.  Each model has multiple systems that are used for different years in the later part of the record.  The climatology from the corresponding system will be used to remove the lead dependent climatology and the newest system will be used for the climatological period

In [4]:
with open("../../../DATA_DOWNLOAD/system_year_info.yaml") as f:
    model_info = yaml.safe_load(f)

#### Set up file info

In [5]:
# directory containing CDS downloads
basepath = "/glade/campaign/cgd/cas/islas/DATASETS/CDS/seasonal/"
# path to store the output
outpath="/glade/campaign/cgd/cas/islas/DATASETS/CDS/seasonal/"
# initialization month
init_mon = 11
# forecast months (NDJFM)
fc_mon = [11,12,1,2,3] 
# variable to sort out (our naming convention)
var='sst'
# the variable name in the raw downloaded files
data_var='sst'

#### Loop over models to sort

In [6]:
models=['UKMO']

In [9]:
dask.config.set({"array.chunk-size":"256MiB"})
chunks={"number": 28,
        "time": 1,
        "step": 1,
        "latitude": 180,
        "longitude": 360
       }


for imodel in models:
    info = model_info[imodel]
    systems = info['forecast'].keys()

    #!!!
    #systems = list(systems)
    #systems = systems[len(systems)-1:len(systems)]
    for isystem in systems:
        print('system=',isystem)
        
        datdir=basepath+imodel+'/grib/'+var+'/'+str(isystem)+'/'
        outdir=outpath+imodel+'/nc/'+var+'/'+str(isystem)+'/'
        os.makedirs(outdir, exist_ok=True)
        
        # loop over forecast months
        alldat_hc=[]
        alldat_fc=[]
        for imon in fc_mon:
            
            print('month=',imon)
            # hindcast and forecast data
            dat_hc = xr.open_mfdataset(datdir+'hindcast_'+var+'_init'+str(init_mon).zfill(2)+'_mon'+str(imon).zfill(2)+'.grib', chunks=chunks)
            dat_fc = xr.open_mfdataset(datdir+'forecast_'+var+'_init'+str(init_mon).zfill(2)+'_mon'+str(imon).zfill(2)+'.grib', chunks=chunks)
            
            #dat_hc = dat_hc.chunk("auto")
            #dat_fc = dat_fc.chunk("auto")
            #dat_hc = dat_hc.chunk({'longitude':30, 'latitude':30})
            #dat_fc = dat_fc.chunk({'longitude':30, 'latitude':30})

            # add time axis in the case there's only one forecast
            if "time" not in dat_fc.dims:
                dat_fc = dat_fc.expand_dims(time=[dat_fc.valid_time.values])
            
            # setting valid_time to be 1st of the following month to deal with valid_time inconsistencies across initializations
            nextmon = imon + 1
            if nextmon > 12:
                nextmon = 1
            year_hc = dat_hc.valid_time.dt.year
            day_1_hc = (year_hc.astype(str)+'-'+str(nextmon).zfill(2)+'-01')
            year_fc = dat_fc.valid_time.dt.year
            day_1_fc = (year_fc.astype(str)+'-'+str(nextmon).zfill(2)+'-01')
            
            dat_hc['valid_time'] = day_1_hc.astype("datetime64[ns]")
            dat_fc['valid_time'] = day_1_fc.astype("datetime64[ns]")
            
            # using sort_predictions function in utils
            dat_monthly_hc = cds_sort.sort_predictions(dat_hc, data_var)
            dat_monthly_fc = cds_sort.sort_predictions(dat_fc, data_var)       

            # do some renaming of dimensions and transpose
            dat_monthly_hc = dat_monthly_hc.rename({data_var:var, 'latitude':'lat', 'longitude':'lon'})
            dat_monthly_hc = dat_monthly_hc.transpose('member','time','lat','lon')

            dat_monthly_fc = dat_monthly_fc.rename({data_var:var, 'latitude':'lat', 'longitude':'lon'})
            dat_monthly_fc = dat_monthly_fc.transpose('member','time','lat','lon')

            # Give it a dimension that corrresponds to the initialization year, instead of time
            if imon == fc_mon[0]: # note this won't necessarily work if you decide not to output the first month
                years_hc = dat_monthly_hc.time.dt.year
                years_fc = dat_monthly_fc.time.dt.year
            dat_monthly_hc['time'] = years_hc
            dat_monthly_hc = dat_monthly_hc.rename(time='year')
            dat_monthly_fc['time'] = years_fc
            dat_monthly_fc = dat_monthly_fc.rename(time='year')

            if "step" in dat_monthly_hc:
                dat_monthly_hc = dat_monthly_hc.drop_vars("step")
            if "step" in dat_monthly_fc:
                dat_monthly_fc = dat_monthly_fc.drop_vars("step")
            
            if 'member' in dat_monthly_hc.valid_time.dims:
                times_hc = dat_monthly_hc.valid_time.isel(member=0).rename('time')
            else:
                times_hc = dat_monthly_hc.valid_time.rename('time')
            if 'member' in dat_monthly_fc.valid_time.dims:
                times_fc = dat_monthly_fc.valid_time.isel(member=0).rename('time')
            else:
                times_fc = dat_monthly_fc.valid_time.rename('time')
            

            datout_hc = xr.merge([dat_monthly_hc, times_hc])
            datout_fc = xr.merge([dat_monthly_fc, times_fc])
            
            alldat_hc.append(datout_hc)
            alldat_fc.append(datout_fc)

            
        alldat_hc = xr.concat(alldat_hc, dim='lead')
        alldat_hc['lead'] = np.arange(1,len(fc_mon)+1,1)
        alldat_hc = alldat_hc.transpose('member','lead','year','lat','lon')

        alldat_fc = xr.concat(alldat_fc, dim='lead')
        alldat_fc['lead'] = np.arange(1,len(fc_mon)+1,1)
        alldat_fc = alldat_fc.transpose('member','lead','year','lat','lon')

        alldat_hc.to_netcdf(outdir+'hindcast_withclim_'+var+'.nc')
        alldat_fc.to_netcdf(outdir+'forecast_withclim_'+var+'.nc')
        
        del alldat_hc, alldat_fc
        del dat_hc, dat_fc
        del dat_monthly_hc, dat_monthly_fc
        del datout_hc, datout_fc
        del times_hc, times_fc
        gc.collect()

system= 12
month= 11
month= 12
month= 1
month= 2
month= 3
system= 13
month= 11
month= 12
month= 1
month= 2
month= 3
system= 14
month= 11
month= 12
month= 1
month= 2
month= 3
system= 15
month= 11
month= 12
month= 1
month= 2
month= 3
system= 600
month= 11
month= 12
month= 1
month= 2
month= 3
system= 601
month= 11
month= 12
month= 1
month= 2
month= 3
system= 602
month= 11
month= 12
month= 1
month= 2
month= 3
system= 603
month= 11
month= 12
month= 1
month= 2
month= 3
system= 604
month= 11
month= 12
month= 1
month= 2
month= 3


In [10]:
cluster.close()

In [8]:
dat_hc

<xarray.Dataset> Size: 167MB
Dimensions:     (number: 28, time: 23, latitude: 180, longitude: 360)
Coordinates:
  * number      (number) int64 224B 0 1 2 3 4 5 6 7 ... 20 21 22 23 24 25 26 27
  * time        (time) datetime64[ns] 184B 1993-11-01 1994-11-01 ... 2015-11-01
  * latitude    (latitude) float64 1kB 89.0 88.0 87.0 86.0 ... -88.0 -89.0 -90.0
  * longitude   (longitude) float64 3kB 0.0 1.0 2.0 3.0 ... 357.0 358.0 359.0
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
    valid_time  (time) datetime64[ns] 184B dask.array<chunksize=(1,), meta=np.ndarray>
Data variables:
    sst         (number, time, latitude, longitude) float32 167MB dask.array<chunksize=(28, 1, 180, 360), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             egrr
    GRIB_centreDescription:  U.K. Met Office - Exeter
    GRIB_subCentre:          98
    Conventions:             CF-1.7
    institution:             U.K. Met Office - Exeter
    history:                 2026-07-26T21:35 GRIB to CDM+CF via cfgrib-0.9.1...